# 5. GPT-2 inference with metrics


In [1]:
# Dynamicly load evaluation metrics and the model
%run -i ../src/models/evaluation.py
%run -i ../src/models/detoxGPT2.py

In [2]:
import numpy as np
import pandas as pd
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
detoxGPT = detoxGPT2()

# Instantiate metric classes
similarity = Similarity()
toxicity = STAToxic()

In [4]:
prompt = "What a fucking stupid thing to say!"

In [ ]:
# Pack the suggestions into a dataframe
df = pd.DataFrame(
    detoxGPT.get_detoxed_suggestions(prompt, max_length=len(prompt), device=DEVICE), columns=["suggestion"]
)

# Add empty column for each metric
metrics = ["wo", "cs", "bleu"]
df[metrics] = pd.DataFrame([[0] * len(metrics)], index=df.index, dtype=float)

# Generate toxicity report for each suggestion
toxicity_report = toxicity.toxicity_report(df["suggestion"])

for index, row in df.iterrows():
    df.loc[index, "wo"] = similarity.get_wo_score(prompt, row["suggestion"])
    df.loc[index, "cs"] = similarity.get_cosine_score(prompt, row["suggestion"])
    df.loc[index, "bleu"] = similarity.get_bleu_score(prompt, row["suggestion"])

# Concat with toxicity report
df = pd.concat([df, toxicity_report], axis=1)
df

In [6]:
# Calculate the score
metric_weights = {"wo": 0.1, "cs": 0.5, "bleu": 0.4}

# Toxicity report should be as low as possible
# Similarity metrics should be as high as possible
df["detox_score"] = 1 - np.mean(df[["toxic"]], axis=1)
df["similarity"] = df[metrics].dot(pd.Series(metric_weights))

# Final score
df["score"] = df[["detox_score", "similarity"]].mean(axis=1)

# Sort by score
df = df.sort_values(by=["score"], ascending=False)
df

,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate,detox_score,similarity,score
2,what a bad thing to say!,0.444444,0.964366,5.025524e-01,0.038117,0.001910,0.007144,0.000731,0.012488,0.001872,0.961883,0.727648,0.844766
3,what a nasty thing to say!,0.444444,0.976615,5.112538e-01,0.171508,0.002237,0.018131,0.000611,0.067049,0.001530,0.828492,0.737254,0.782873
1,That's too bad to say!,0.090909,0.834366,2.516173e-01,0.084930,0.002051,0.006438,0.000543,0.015024,0.001064,0.915070,0.526921,0.720996
4,"I'm pretty sure you meant the word """"blonde,""""...",0.000000,0.698961,1.747629e-78,0.002952,0.000007,0.000428,0.000011,0.000368,0.000010,0.997048,0.349481,0.673265
5,it,0.000000,0.590194,1.243510e-238,0.037838,0.001227,0.005018,0.000238,0.006115,0.000733,0.962162,0.295097,0.628630
0,"Oh my God, how dare you say that!",0.071429,0.746313,1.435887e-01,0.595142,0.037728,0.117315,0.000883,0.049845,0.002769,0.404858,0.437735,0.421296


In [7]:
# Print the suggestion with the highest score
suggestion = df.iloc[0]["suggestion"]
print(f"{prompt} -> {suggestion}")

What a fucking stupid thing to say! -> what a bad thing to say!
